# UMP2 Test

In this notebook, we will test the UMP2 implementation. It must be noted that neither Pyscf nor us obtain converged UHF wave functions. However, the unconverged result is the same. 

## Closed shell UMP2

In [1]:
from pyscf import gto, scf, mp
from py_mods.src.SCF.CSUHF import CS_UHF
from py_mods.src.MP2.CSMP2 import CS_MP2
from py_mods.src.SCF.external import UHF_context_from_pyscf

In [2]:
pyscf_args = {
    "atom": "Ne 0 0 0; Ne 0 0 1",
    "spin": 0,
    "charge": 0,
    "basis": "cc-pvtz",
}

mol = gto.M(**pyscf_args)

mf = scf.UHF(mol)
mf.conv_tol = 1e-14
mf.conv_tol_grad = 1e-14
mf.max_cycle = 200
e_He = mf.kernel()
e_elec = mf.energy_elec()[0]
e_nuc = mol.energy_nuc()

print(f"Pyscf UHF energy: {e_He}")
print(f"Pyscf UHF electronic energy: {e_elec}")

SCF not converged.
SCF energy = -255.620961295416 after 200 cycles  <S^2> = -3.5527137e-15  2S+1 = 1
Pyscf UHF energy: -255.62096129541553
Pyscf UHF electronic energy: -308.5386823874155


In [3]:
UHF_cxt = UHF_context_from_pyscf(**pyscf_args)
UHF_cxt.break_symm = True
UHF_cxt.p_guess = 'RHF'

UHF_res = CS_UHF(UHF_cxt)

print(f"\nSCF energy: {UHF_res.E_UHF} (converged: {UHF_res.converged})")
print(f"SCF pyscf: {e_elec}")
print(f"Difference: {UHF_res.E_UHF.real - e_elec} \n")


SCF energy: (-308.53868238741586+0j) (converged: True)
SCF pyscf: -308.5386823874155
Difference: -3.410605131648481e-13 



In [4]:
mymp = mp.UMP2(mf).run()
mp_results = CS_MP2(UHF_res)

E(UMP2) = -256.206724567632  E_corr = -0.585763272216132
E(SCS-UMP2) = -256.184456589529  E_corr = -0.563495294113252


In [5]:
print(f"\n\nMP2 calc: {mp_results.E_MP2}, E_corr = {mp_results.E_corr}")
print(f"MP2 pyscf: {mf.energy_elec()[0] + mymp.e_corr}, E_corr = {mymp.e_corr}")
print(
    f"Differences: {mp_results.E_MP2 - mf.energy_elec()[0] - mymp.e_corr}, E_corr = {mp_results.E_corr - mymp.e_corr}\n"
)



MP2 calc: (-309.124445659632+0j), E_corr = (-0.585763272216118-0j)
MP2 pyscf: -309.12444565963165, E_corr = -0.5857632722161323
Differences: (-3.9257486150745535e-13+0j), E_corr = (1.432187701766452e-14-0j)



## Open shell MP2 

In [6]:
pyscf_args = {
    "atom": "Ne 0 0 0; N 0 0 1",
    "spin": 1,
    "charge": 0,
    "basis": "cc-pvtz",
}

mol = gto.M(**pyscf_args)

mf = scf.UHF(mol)
mf.conv_tol = 1E-14
mf.conv_tol_grad = 1E-14
mf.max_cycle = 200
e_He = mf.kernel()
e_elec = mf.energy_elec()[0]
e_nuc = mol.energy_nuc()

print(f"Pyscf UHF energy: {e_He}")
print(f"Pyscf UHF electronic energy: {e_elec}")

SCF not converged.
SCF energy = -182.323301435604 after 200 cycles  <S^2> = 0.75868668  2S+1 = 2.0086679
Pyscf UHF energy: -182.32330143560424
Pyscf UHF electronic energy: -219.36570620000424


In [7]:
UHF_cxt = UHF_context_from_pyscf(**pyscf_args)
UHF_cxt.break_symm = True
UHF_cxt.p_guess = 'RHF'

UHF_res = CS_UHF(UHF_cxt)

print(f"\nSCF energy: {UHF_res.E_UHF} (converged: {UHF_res.converged})")
print(f"SCF pyscf: {e_elec}")
print(f"Difference: {UHF_res.E_UHF.real - e_elec} \n")


SCF energy: (-219.36570620000438+0j) (converged: True)
SCF pyscf: -219.36570620000424
Difference: -1.4210854715202004e-13 



In [8]:
mymp = mp.UMP2(mf).run()
mp_results = CS_MP2(UHF_res)

E(UMP2) = -182.754905321944  E_corr = -0.43160388633959
E(SCS-UMP2) = -182.745197954619  E_corr = -0.421896519014422


In [9]:
print(f"\n\nMP2 calc: {mp_results.E_MP2}, E_corr = {mp_results.E_corr}")
print(f"MP2 pyscf: {mf.energy_elec()[0] + mymp.e_corr}, E_corr = {mymp.e_corr}")
print(
    f"Differences: {mp_results.E_MP2 - mf.energy_elec()[0] - mymp.e_corr}, E_corr = {mp_results.E_corr - mymp.e_corr}\n"
)



MP2 calc: (-219.79731008634397+0j), E_corr = (-0.43160388633958047-0j)
MP2 pyscf: -219.79731008634383, E_corr = -0.43160388633959024
Differences: (-1.3933298959045715e-13+0j), E_corr = (9.769962616701378e-15-0j)

